# DentHub (CC26-PSU285) – Data Wrangling, Cleaning, Feature Engineering & EDA

This notebook documents the complete data science workflow required by `docs/ARCHITECTURE.md §11` and `docs/PRD.md §16`: multi-source dental image gathering, quality assessment, manual cleaning, perceptual-hash deduplication, stratified group splitting **before** augmentation, engineered image-quality features, and explanatory charts backing every conclusion.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROCESSED = Path('../data/processed')
CHARTS = Path('../outputs/charts')
assessment = json.loads((PROCESSED / 'assessment_report.json').read_text(encoding='utf-8'))
features_df = pd.read_csv(PROCESSED / 'engineered_features.csv')
print('Assessment summary:', json.dumps(assessment, indent=2))
features_df.head()

## 1. Data Gathering, Cleaning & Anti-Leakage Verification

- Raw files scanned (included sources): **14249**
- Pre-augmented & duplicate YOLO export files removed: **6757**
- Off-target clinical classes dropped (`hypodontia`, `Mouth Ulcer`, `Tooth Discoloration`): **3241**
- Corrupt files dropped: **0**
- Low-res / extreme aspect dropped: **18**
- Cross-dataset exact & perceptual near-duplicates removed (Hamming ≤ 4): **2704**
- Final clean unique images: **1302** (train=910, val=196, test=196)

**Conclusion 1:** Perceptual hashing eliminated duplicate images across sources before splitting, preventing inflated validation/test accuracy. Every split has balanced class coverage as shown below.

In [ ]:
display(Image(filename=str(CHARTS / '01_class_distribution_by_split.png')))
display(Image(filename=str(CHARTS / '02_source_composition.png')))

## 2. Feature Engineering & Quality-Gate Threshold Justification

- Median mean luminance: **129.3** (5th–95th percentile: 84.2–174.4)
- Median Laplacian sharpness: **301.7**
- Mean CLAHE contrast gain: **4.39**

**Conclusion 2:** Real-world dental photos span wide luminance and sharpness ranges. Setting serving quality-gate thresholds at luminance `[35, 235]` and Laplacian variance `≥ 30` rejects dark/blurry captures without discarding well-lit screening photos, while CLAHE improves local enamel/gingiva contrast consistently across classes.

In [ ]:
display(Image(filename=str(CHARTS / '03_quality_features_by_class.png')))
display(Image(filename=str(CHARTS / '04_clahe_contrast_and_feature_correlation.png')))